# Lab 3 / Mission 3 — Mine Your First Block

Build a Merkle tree over a list of transactions, then mine a simplified block
header by searching for a nonce that satisfies a difficulty prefix.

**Serialization rules** (documented so a checker can reproduce every field):

```
leaf hash  = sha256(utf8(transaction))
inner hash = sha256(utf8(left_hex + right_hex))
odd level  = last hash duplicated (Bitcoin convention)
header     = prev_hash + merkle_root + student_id + str(nonce)
block hash = sha256(utf8(header))
```

## Setup and Merkle tree construction

In [ ]:
import hashlib
import json

STUDENT_ID = "123456"          # replace with the real student ID before submitting
DIFFICULTY = "0000"            # block hash must start with this prefix
PREV_HASH = "0" * 64           # genesis block: no predecessor

TRANSACTIONS = [
    "alice->bob:10",
    "bob->carol:5",
    "carol->dave:2",
    "dave->alice:1",
    "alice->eve:7",
]


def sha256_hex(data: str) -> str:
    return hashlib.sha256(data.encode("utf-8")).hexdigest()


# ---------------------------------------------------------------- merkle tree
def merkle_root(transactions: list[str], verbose: bool = False) -> str:
    """Build a Merkle root; returns the single remaining hash."""
    if not transactions:
        raise ValueError("cannot build a Merkle tree over an empty list")

    level = [sha256_hex(tx) for tx in transactions]
    if verbose:
        print("Level 0 (leaf hashes):")
        for tx, h in zip(transactions, level):
            print(f"  {tx:18} -> {h}")

    depth = 0
    while len(level) > 1:
        depth += 1
        if len(level) % 2 == 1:
            level = level + [level[-1]]          # duplicate the last hash
        level = [
            sha256_hex(level[i] + level[i + 1])
            for i in range(0, len(level), 2)
        ]
        if verbose:
            print(f"Level {depth}:")
            for h in level:
                print(f"  {h}")

    return level[0]


# --------------------------------------------------------------------- mining
def build_header(prev_hash: str, root: str, student_id: str, nonce: int) -> str:
    return f"{prev_hash}{root}{student_id}{nonce}"


def mine(prev_hash: str, root: str, student_id: str, difficulty: str):
    """Increment the nonce until the block hash starts with `difficulty`."""
    nonce = 0
    while True:
        header = build_header(prev_hash, root, student_id, nonce)
        block_hash = sha256_hex(header)
        if block_hash.startswith(difficulty):
            return nonce, block_hash
        nonce += 1

## Steps 1–3 — build the Merkle root

In [ ]:
root = merkle_root(TRANSACTIONS, verbose=True)
print('\nMerkle root:', root)

## Steps 4–5 — mine the block

Increment the nonce until the block hash starts with the difficulty prefix.

In [ ]:
nonce, block_hash = mine(PREV_HASH, root, STUDENT_ID, DIFFICULTY)
print('nonce      :', nonce)
print('block hash :', block_hash)
print('attempts   :', nonce + 1, '| expected ~', 16 ** len(DIFFICULTY))

## Verification — exactly what the checker will redo

In [ ]:
assert merkle_root(TRANSACTIONS) == root
assert sha256_hex(build_header(PREV_HASH, root, STUDENT_ID, nonce)) == block_hash
assert block_hash.startswith(DIFFICULTY)
print('root recomputes, nonce reproduces the hash, difficulty satisfied')

## Tamper test — change one transaction

Changing a single transaction changes the Merkle root, which changes the block
header, which invalidates the nonce. The block must be re-mined.

In [ ]:
tampered = list(TRANSACTIONS)
tampered[1] = 'bob->carol:500'
tampered_root = merkle_root(tampered)
print('original root :', root)
print('tampered root :', tampered_root)
tampered_hash = sha256_hex(build_header(PREV_HASH, tampered_root, STUDENT_ID, nonce))
print('block hash with the old nonce:', tampered_hash)
print('still valid?', tampered_hash.startswith(DIFFICULTY))